# Car Price Prediction

This notebook demonstrates a complete machine learning pipeline for predicting car prices.

## STEP 1 — LIBRARY IMPORTS

In [ ]:
import os
import glob
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## STEP 2 — LOAD DATASET
Auto-detecting the CSV file in the current directory.

In [ ]:
# Auto-detect CSV file
csv_files = glob.glob("*.csv")
if not csv_files:
    raise FileNotFoundError("No CSV file found in the current directory.")

data_file = csv_files[0]
print(f"Loading dataset: {data_file}")

df = pd.read_csv(data_file)
display(df.shape)
display(df.head())
display(df.dtypes)
display(df.describe())

## STEP 3 — EXPLORATORY DATA ANALYSIS
Understanding the distribution of data and relationships between features.

In [ ]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

In [ ]:
# Target variable distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Selling_Price'], kde=True, color='blue')
plt.title('Distribution of Selling Price')
plt.xlabel('Selling Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Categorical variables
cat_cols = ['Fuel_Type', 'Selling_type', 'Transmission', 'Owner']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(cat_cols):
    ax = axes[i//2, i%2]
    sns.countplot(x=col, data=df, ax=ax, palette='Set2')
    ax.set_title(f'Count of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Price vs Categorical Variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(cat_cols):
    ax = axes[i//2, i%2]
    sns.boxplot(x=col, y='Selling_Price', data=df, ax=ax, palette='Set2')
    ax.set_title(f'Selling Price vs {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots for numerical variables
num_cols = ['Year', 'Present_Price', 'Driven_kms']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(num_cols):
    sns.scatterplot(x=col, y='Selling_Price', data=df, ax=axes[i], color='teal', alpha=0.6)
    axes[i].set_title(f'Selling Price vs {col}')
plt.tight_layout()
plt.show()

## STEP 4 — DATA PREPROCESSING
Handling categorical variables, creating new features, and scaling.

In [ ]:
# Create Age of car feature
current_year = 2024
df['Car_Age'] = current_year - df['Year']

# Drop Car_Name and Year as they are not needed for modeling directly
df_model = df.drop(['Car_Name', 'Year'], axis=1)

# Encode categorical variables
label_cols = ['Fuel_Type', 'Selling_type', 'Transmission']
le = LabelEncoder()
for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Display correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df_model.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Features and Target
X = df_model.drop('Selling_Price', axis=1)
y = df_model['Selling_Price']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## STEP 5 — MODEL TRAINING & EVALUATION
Training multiple models and comparing their performance.

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

# Train and evaluate models
results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
display(results_df)

In [ ]:
# Visualize model performance
plt.figure(figsize=(10, 6))
sns.barplot(x='R2 Score', y='Model', data=results_df, palette='viridis')
plt.title('Model Comparison based on R2 Score')
plt.xlabel('R2 Score')
plt.ylabel('Model')
plt.show()

## STEP 6 — HYPERPARAMETER TUNING
Fine-tuning the best model (typically Random Forest or Gradient Boosting) using GridSearchCV.

In [ ]:
# Tune Random Forest as an example
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42),
                           param_grid=param_grid,
                           cv=5, n_jobs=-1, scoring='r2', verbose=1)

grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV R2 Score:", grid_search.best_score_)

# Evaluate on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test_scaled)
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred_best):.4f}")
print(f"Test R2: {r2_score(y_test, y_pred_best):.4f}")

## CONCLUSION
The notebook demonstrates data loading, EDA, preprocessing, and model building for car price prediction. The optimal model achieves a high R2 score, indicating strong predictive performance on the test data.